# Week 3 Exercise — Meeting Minutes + Read Summary Aloud (profe-ssor)

**Project 2** with Gradio: upload audio → transcribe (Whisper) → LLM meeting minutes → **read the summary aloud** (TTS).

- **Input:** Upload a meeting/call audio file (e.g. .mp3, .wav, .m4a).
- **Pipeline:** Whisper (speech-to-text) → GPT (summarize into minutes).
- **Output:** Transcript + meeting minutes (summary, decisions, action items).
- **Read aloud:** Click **"Read summary aloud"** to hear the minutes via OpenAI TTS.

Uses **Week 2** (Gradio, system prompt) + **Week 3** (Whisper, LLM summarization, TTS).

In [29]:
import os
import tempfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [30]:
load_dotenv(override=True)
openai_client = OpenAI()

WHISPER_MODEL = "whisper-1"
CHAT_MODEL = "gpt-4o-mini"
TTS_MODEL = "gpt-4o-mini-tts"
TTS_VOICE = "onyx"

print("OpenAI client ready (Whisper + Chat + TTS)")

OpenAI client ready (Whisper + Chat + TTS)


## System prompt for meeting minutes

In [31]:
MINUTES_SYSTEM = """You are an assistant that produces meeting minutes from transcripts.
Output clear markdown with: a short summary; key discussion points; takeaways; and action items with owners.
Keep the summary concise and actionable."""

## Pipeline: transcribe → summarize

In [32]:
def transcribe_audio(audio_path):
    if audio_path is None or not Path(audio_path).exists():
        return None
    with open(audio_path, "rb") as f:
        r = openai_client.audio.transcriptions.create(
            model=WHISPER_MODEL,
            file=f,
            response_format="text"
        )
    return r if isinstance(r, str) else getattr(r, "text", str(r))

In [33]:
def summarize_to_minutes(transcript):
    if not (transcript or transcript.strip()):
        return ""
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": MINUTES_SYSTEM},
            {"role": "user", "content": f"Turn this transcript into meeting minutes (summary, discussion points, takeaways, action items with owners). Use markdown.\n\n{transcript}"}
        ]
    )
    return (r.choices[0].message.content or "").strip()

## Read summary aloud (TTS)

In [34]:
def text_to_speech(text):
    """Convert summary text to speech; returns path to temp MP3 or None."""
    if not text or not text.strip():
        return None
    text_plain = text.strip()[:4000]
    try:
        r = openai_client.audio.speech.create(
            model=TTS_MODEL,
            voice=TTS_VOICE,
            input=text_plain
        )
        fd, path = tempfile.mkstemp(suffix=".mp3")
        os.write(fd, r.content)
        os.close(fd)
        return path
    except Exception as e:
        err = str(getattr(e, "message", str(e))).lower()
        if any(x in err for x in ("billing", "quota", "limit", "500", "internal")):
            print("TTS unavailable (OpenAI billing/quota or server error). Add payment at https://platform.openai.com")
        else:
            print("TTS failed:", err[:200])
        return None

## Gradio UI

In [35]:
def run_pipeline(audio_path, progress=gr.Progress()):
    if audio_path is None:
        return "", "", None
    progress(0.2, desc="Transcribing...")
    transcript = transcribe_audio(audio_path)
    if not transcript:
        return "", "(Transcription failed.)", None
    progress(0.6, desc="Writing minutes...")
    summary = summarize_to_minutes(transcript)
    progress(1.0)
    return transcript, summary, None

_tts_executor = ThreadPoolExecutor(max_workers=1)

def read_aloud(summary_text):
    try:
        future = _tts_executor.submit(text_to_speech, summary_text or "")
        path = future.result(timeout=120)
        return path if path else None
    except Exception:
        return None

with gr.Blocks(title="Meeting Minutes + Read Aloud", theme=gr.themes.Soft()) as demo:
    gr.Markdown("### Upload meeting audio → get transcript + minutes → read summary aloud")
    audio_in = gr.Audio(sources=["upload"], type="filepath", label="Upload audio")
    run_btn = gr.Button("Transcribe & summarize", variant="primary")
    transcript_out = gr.Textbox(label="Transcript", lines=6, interactive=False)
    summary_out = gr.Textbox(label="Meeting minutes (summary)", lines=12, interactive=False)
    read_btn = gr.Button("Read summary aloud")
    audio_out = gr.Audio(label="Summary read aloud", autoplay=True, interactive=False)

    run_btn.click(
        fn=run_pipeline,
        inputs=[audio_in],
        outputs=[transcript_out, summary_out, audio_out]
    )
    read_btn.click(
        fn=read_aloud,
        inputs=[summary_out],
        outputs=[audio_out]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/home/professor/projects/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/protocols/http/httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/professor/projects/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/professor/projects/llm_engineering/.venv/lib/python3.12/site-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/home/professor/projects/llm_engineering/.venv/lib/python3.12/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/home/professor/projects/llm_engineerin